# Embeddings
## chargement des modèles
### modèle paraphrase: chargement + embeddings

In [1]:
### Chargement des données processées

import pandas as pd
main_data = "data_final_streamlit"
temp_save = "data_2epoch"
df = pd.read_csv(f'{main_data}/40k_final_process.csv', sep=',', header=0, index_col=0) # chargement des données après preprocessing final

In [7]:
from sentence_transformers import SentenceTransformer

model_paraphrase = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

embeddings_paraphrase = model_paraphrase.encode(df['commentaire'].tolist(), normalize_embeddings=True)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2292.51it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### fine-tuning paraphrase par LoRA

In [3]:
from peft import LoraConfig, get_peft_model

"""
Low Rank Adaptation (LoRA) consiste à :
_ geler les poids du modèle pré-entrainé
_ ajouter de petites matrices de rang faible dans certaines couches 
_ n'entrainer que ces matrices
Ce qui permet :
_ beaucoup moins de paramètres à entrainer
_ et donc un entrainement plus rapide et moins coûteux

"""


model = model_paraphrase

# backbone transformer
transformer = model._first_module().auto_model

# config LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

# inject LoRA
transformer = get_peft_model(transformer, lora_config)

# ⚡ FIX : FORCER return_dict=False pour compatibilité SentenceTransformers
transformer.config.return_dict = False

# réinjection dans SentenceTransformer
model._first_module().auto_model = transformer

### Générer des paires positives (all this part can be skiped and charged later on notebook)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

"""
pour permettre le fine-tuning en l'abscence de labels, on fait du self-supervised /contrastive learning.
l'idée c'est de créer artificiellement à partir du dataset initial des paires positives (qui jouent le role de labels).
Les autre paires seront considérés comme négatives. Le modèle apprend en même temps: ces phrases sont similaires, 
ces autres phrases sont différentes. On utilise pour celà une loss contrastive pour l'entrainement.
Pour générer des paires positives, on choisit la méthode de backtranslation: traduire les commentaires depuius le français
vers l'anglais puis de nouveau vers le français pour obtenir une variabilité grammaticale et orthographique 
tout en conservant un sens commun. En effet les modèles de traduction apprennent une représentation sémantique; la
reconstruction introduit donc des variations naturelles.

"""

# modèles de traduction
model_name_fr_en = "Helsinki-NLP/opus-mt-fr-en"
model_name_en_fr = "Helsinki-NLP/opus-mt-en-fr"

tokenizer_fr_en = MarianTokenizer.from_pretrained(model_name_fr_en)
model_fr_en = MarianMTModel.from_pretrained(model_name_fr_en)

tokenizer_en_fr = MarianTokenizer.from_pretrained(model_name_en_fr)
model_en_fr = MarianMTModel.from_pretrained(model_name_en_fr)

In [ ]:
def back_translate(text):
    if not text or len(text.strip()) == 0:
        return text

    # FR → EN
    batch = tokenizer_fr_en(
        [text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    translated = model_fr_en.generate(**batch)
    en_text = tokenizer_fr_en.batch_decode(translated, skip_special_tokens=True)[0]

    # EN → FR
    batch = tokenizer_en_fr(
        [en_text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    translated = model_en_fr.generate(**batch)
    fr_text = tokenizer_en_fr.batch_decode(translated, skip_special_tokens=True)[0]

    return fr_text

In [ ]:
from sentence_transformers import util

"""
Parfois la backtranslation dénature trop le sens. Pour éviter d'injecter du bruit (signal d'apprentissage erroné) dans les paires positives,
il est judicieux d'utiliser un autre modèle pour contrôler la similarité sémantique. Celà permet d'obtenir
des paires positives fiables. L'utilisation d'un autre modèle permet d'éviter le phénomène de biais circulaires
(pour que le modèle ne valide pas ses propres erreurs).

"""

model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
#model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')

def is_good_pair(text, augmented, threshold=0.7): # le seuil est fixé de manière empirique pour ne pas perdre trop de diversité tout en évitant de générer du bruit.
    emb1 = model.encode(text, normalize_embeddings=True)
    emb2 = model.encode(augmented, normalize_embeddings=True)
    
    score = util.cos_sim(emb1, emb2)
    return score > threshold

In [ ]:

filtered_pairs = []

long_texts = df.loc[df['length_comm'] > 75]
short_texts = df.loc[df['length_comm'] <= 75]



total_sample_size = int(len(df) * 0.3)

"""
comme on sait que le dataset est très asymétrique, on s'assure d'avoir une représentation de tous les types d'avis dans 
le train.
"""

# Répartition 50% longs, 50% courts
long_sample_size = total_sample_size // 2
short_sample_size = total_sample_size - long_sample_size

long_sample = long_texts.sample(n=long_sample_size, random_state=42)
short_sample = short_texts.sample(n=short_sample_size, random_state=42)

df_sample = pd.concat([long_sample, short_sample]).sample(frac=1, random_state=42)

# convertir en liste
corpus_sample = df_sample['commentaire'].to_list()

print(f"Taille du corpus échantillonné : {len(corpus_sample)}")

for text in corpus_sample:
    aug = back_translate(text)
    
    if is_good_pair(text, aug):
        filtered_pairs.append((text, aug))

In [ ]:
""" exporter filtered pairs pour être sûr de ne pas les perdre
Effectivement le calcul des filtered positive pairs est excessivement long.
Pour eviter des calculs inutiles, on exporte les paires calculées.

"""
import csv

# filtered_pairs = [('texte1', 'texte2'), ('texte3', 'texte4'), ...]
with open("filtered_pairs.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # optionnel : écrire un header
    writer.writerow(["text1", "text2"])
    writer.writerows(filtered_pairs)

### Import des "filtered pairs" pour fine-tuning LoRA du modèle paraphrase

In [ ]:
import csv

filtered_pairs = []

with open("filtered_pairs.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    
    next(reader)  # skip header
    
    for row in reader:
        text1, text2 = row
        filtered_pairs.append((text1, text2))

### création du train set

In [ ]:
from sentence_transformers import InputExample

train_examples = [
    InputExample(texts=[t1, t2])
    for t1, t2 in filtered_pairs
]

In [ ]:
from sentence_transformers import losses

"""
une loss comme MultipleNegativesRankingLoss repose sur:
_une paire positive (ancre)
_des négatifs implicites (tous les autres éléments du batch)
Le modèle apprend une géométrie de l'espace qui rapproche les positifs et éloigne les négatifs.
"""

train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_examples,
    batch_size=32,
    shuffle=True
)

In [ ]:
warmup_steps = int(len(train_dataloader) * 0.1)

### fine-tuning du modèle

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10, # des tests ont été fait avec 1, 2 et 5 epoch mais ils performaient moins bien que le paraphrase natif pour le clustering sémantique évalué par silouhette score
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    use_amp=True
)

In [ ]:
model.save('paraphrase_lora_10epoch')

### Import d'un modèle préalablement fine-tuné

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel
from sentence_transformers import SentenceTransformer, models

"""
L'étape de fine-tuning est également très longue et gourmande en ressources de calcul. 
Il est donc possible de recharger les modèles déjà fine-tunés à partir de cette étape.
"""


tokenizer = AutoTokenizer.from_pretrained(
"sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)
base_model = AutoModel.from_pretrained(
"sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

model = PeftModel.from_pretrained(base_model, "data/paraphrase_lora_2epoch")
model = model.merge_and_unload()
model.save_pretrained(f"{temp_save}/final_lora")
#model = AutoModel.from_pretrained("data/final_lora")

word_embedding_model = models.Transformer(f"{temp_save}/final_lora")

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2296.19it/s, Materializing param=pooler.dense.weight]                               
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3037.48it/s, Materializing param=pooler.dense.weight]                               


In [3]:
# embeddings du modèle fine-tuné avec LoRA

model.max_seq_length = 512
embeddings_paraphrase_lora = model.encode(df['commentaire'].tolist(), normalize_embeddings=True)

# Clustering
## Optimisation  UMAP + HDBSCAN

In [4]:
import numpy as np
from sklearn.metrics import silhouette_score
from itertools import product
from sklearn.cluster import HDBSCAN
from umap import UMAP

def evaluate_clustering(X, labels):
    # enlever le bruit
    mask = labels != -1
    
    if len(set(labels[mask])) < 2:
        return -1  # pas assez de clusters
    
    return silhouette_score(X[mask], labels[mask])

scores_umap = {}

def optimize_umap(embeddings):
    param_grid = {
        "n_neighbors": [5, 10, 15],
        "min_dist": [0.0, 0.05, 0.1],
        "n_components": [5, 10]
    }

    best_score = -1
    best_params = None
    best_embedding = None

    for n_neighbors, min_dist, n_components in product(
        param_grid["n_neighbors"],
        param_grid["min_dist"],
        param_grid["n_components"]
    ):
        umap_model = UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=n_components,
            metric="cosine",
            random_state=42
        )

        X_reduced = umap_model.fit_transform(embeddings)

        # clustering temporaire (valeur fixe)
        clusterer = HDBSCAN(min_cluster_size=30, metric="euclidean")
        labels = clusterer.fit_predict(X_reduced)

        score = evaluate_clustering(X_reduced, labels)
        scores_umap['n_neighbors_' + str(n_neighbors) + "_min_dist_" + str(min_dist) + '_n_components_' + str(n_components)] = score

        if score > best_score:
            best_score = score
            best_params = (n_neighbors, min_dist, n_components)
            best_embedding = X_reduced

    print("Best UMAP:", best_params, "score:", best_score)
    return best_embedding, best_params

scores_hdbscan = {}

def optimize_hdbscan(X_reduced):
    min_cluster_sizes = [10, 20, 30, 50, 100, 150]

    best_score = -1
    best_param = None
    best_labels = None

    for mcs in min_cluster_sizes:
        clusterer = HDBSCAN(
            min_cluster_size=mcs,
            metric="euclidean"
        )

        labels = clusterer.fit_predict(X_reduced)

        score = evaluate_clustering(X_reduced, labels)
        scores_hdbscan['min_cluster_size_' + str(mcs) ] = score

        if score > best_score:
            best_score = score
            best_param = mcs
            best_labels = labels

    print("Best HDBSCAN min_cluster_size:", best_param, "score:", best_score)
    return best_labels, best_param

In [5]:
# 1️⃣ UMAP
X_reduced_lora, best_umap_params_lora = optimize_umap(embeddings_paraphrase_lora)

# 2️⃣ HDBSCAN
labels_lora, best_mcs = optimize_hdbscan(X_reduced_lora)

/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/

Best UMAP: (15, 0.0, 5) score: 0.9352481365203857


/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10.

Best HDBSCAN min_cluster_size: 20 score: 0.9484081268310547


In [8]:
# 1️⃣ UMAP
X_reduced_nolora, best_umap_params_nolora = optimize_umap(embeddings_paraphrase)

# 2️⃣ HDBSCAN
labels_nolora, best_mcs = optimize_hdbscan(X_reduced_nolora)

/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/

Best UMAP: (5, 0.0, 5) score: 0.5811600089073181


/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10.

Best HDBSCAN min_cluster_size: 10 score: 0.6324224472045898


## Optimisation UMAP + KMeans

In [9]:
from sklearn.cluster import KMeans

scores_kmeans = {}

def optimize_umap_kmeans(embeddings):
    param_grid = {
        "n_neighbors": [5, 10, 15],
        "min_dist": [0.0, 0.05, 0.1],
        "n_components": [5, 10]
    }

    best_score = -1
    best_params = None
    best_embedding = None

    for n_neighbors, min_dist, n_components in product(
        param_grid["n_neighbors"],
        param_grid["min_dist"],
        param_grid["n_components"]
    ):
        umap_model = UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=n_components,
            metric="cosine",
            random_state=42
        )

        X_reduced = umap_model.fit_transform(embeddings)

        # clustering temporaire (valeur fixe)
        clusterer = KMeans(n_clusters=10, random_state=42, n_init="auto")
        labels = clusterer.fit_predict(X_reduced)

        score = evaluate_clustering(X_reduced, labels)
        scores_kmeans['n_neighbors_' + str(n_neighbors) + "_min_dist_" + str(min_dist) + '_n_components_' + str(n_components)] = score

        if score > best_score:
            best_score = score
            best_params = (n_neighbors, min_dist, n_components)
            best_embedding = X_reduced
            best_labels = labels

    print("Best UMAP + KMEANS:", best_params, "score:", best_score)
    return best_labels, best_embedding, best_params

In [10]:
labels_lora_kmeans, X_reduced_lora_kmeans, best_umap_params_kmeans_lora = optimize_umap_kmeans(embeddings_paraphrase_lora)



/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism

Best UMAP + KMEANS: (15, 0.0, 5) score: 0.3571191132068634


In [11]:
labels_nolora_kmeans, X_reduced_nolora_kmeans, best_umap_params_kmeans_nolora = optimize_umap_kmeans(embeddings_paraphrase)



/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism

Best UMAP + KMEANS: (10, 0.0, 5) score: 0.44435593485832214


## Agglomerative clustering + post-process du bruit pour UMAP + HDBSCAN

In [12]:
import numpy as np ### calcul des centroids

def compute_centroids(embeddings, labels):
    unique_labels = [l for l in set(labels) if l != -1]  # ignorer le bruit
    centroids = {}
    for l in unique_labels:
        centroids[l] = embeddings[labels == l].mean(axis=0)
    return centroids

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

centroids = compute_centroids(embeddings_paraphrase, labels_nolora)
centroid_keys = list(centroids.keys())
centroid_matrix = np.stack([centroids[k] for k in centroid_keys])
sim_matrix = cosine_similarity(centroid_matrix)

In [14]:
centroids_lora = compute_centroids(embeddings_paraphrase_lora, labels_lora)
centroid_lora_keys = list(centroids_lora.keys())
centroid_lora_matrix = np.stack([centroids_lora[k] for k in centroid_lora_keys])
sim_matrix_lora = cosine_similarity(centroid_lora_matrix)

In [17]:
from sklearn.cluster import AgglomerativeClustering

agglo = AgglomerativeClustering(
    n_clusters=10, 
    metric='cosine', 
    linkage='average'
)
new_cluster_ids = agglo.fit_predict(centroid_matrix)
new_cluster_ids_lora = agglo.fit_predict(centroid_lora_matrix)

In [18]:
agglo_labels_nolora = labels_nolora.copy()
for old_label, new_label in zip(centroid_keys, new_cluster_ids):
    agglo_labels_nolora[labels_nolora == old_label] = new_label

In [19]:
agglo_labels_lora = labels_lora.copy()
for old_label, new_label in zip(centroid_lora_keys, new_cluster_ids_lora):
    agglo_labels_lora[labels_lora == old_label] = new_label

In [20]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def postprocess_clusters(embeddings, labels, similarity_threshold=0.6, verbose=True):
    """
    Nettoie et réassigne les points bruités après HDBSCAN.
    
    Args:
        embeddings (np.ndarray): matrice des embeddings (n_samples x dim)
        labels (np.ndarray): labels HDBSCAN (-1 = bruit)
        similarity_threshold (float): seuil cosine pour réassignation du bruit
        verbose (bool): afficher résumé
    
    Returns:
        np.ndarray: labels post-processed
    """
    
    new_labels = labels.copy()
    
    # indices
    clustered_idx = labels != -1
    noise_idx = labels == -1
    
    if np.sum(noise_idx) == 0:
        if verbose:
            print("Aucun bruit détecté.")
        return new_labels
    
    clustered_emb = embeddings[clustered_idx]
    clustered_labels = labels[clustered_idx]
    noise_emb = embeddings[noise_idx]
    
    # calculer similarité cosine entre bruit et points clusterisés
    sim = cosine_similarity(noise_emb, clustered_emb)
    
    # assigner chaque point au cluster le plus proche si au-dessus du seuil
    nearest_idx = np.argmax(sim, axis=1)
    nearest_sim = np.max(sim, axis=1)
    
    for i, (sim_val, idx) in enumerate(zip(nearest_sim, nearest_idx)):
        if sim_val >= similarity_threshold:
            new_labels[np.where(noise_idx)[0][i]] = clustered_labels[idx]
        # sinon laisser -1 (bruit)
    
    # résumé
    if verbose:
        total_noise = np.sum(labels == -1)
        remaining_noise = np.sum(new_labels == -1)
        print(f"Points bruit initiaux : {total_noise}")
        print(f"Points bruit restant après réassignation : {remaining_noise}")
        print(f"Points réassignés : {total_noise - remaining_noise}")
    
    return new_labels

In [21]:
labels_nolora_agglo_postprocess = postprocess_clusters(embeddings_paraphrase, agglo_labels_nolora)

labels_lora_agglo_postprocess = postprocess_clusters(embeddings_paraphrase_lora, agglo_labels_lora)


Points bruit initiaux : 14675
Points bruit restant après réassignation : 51
Points réassignés : 14624
Points bruit initiaux : 592
Points bruit restant après réassignation : 0
Points réassignés : 592


## Evaluation des Clusters

In [27]:

df_viz = pd.DataFrame({
    "text": df['commentaire'],
    "cluster": labels_lora_agglo_postprocess
})

print(df_viz['cluster'].value_counts())

print(df_viz['cluster'].shape)
#sns.countplot(df_viz.loc[df_viz['cluster'] != -1, 'cluster'])

cluster
1    14941
3     6837
4     6274
2     4463
6     3291
5     1064
0      672
9      631
8       56
7       32
Name: count, dtype: int64
(38261,)


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from sklearn.metrics import silhouette_score
from collections import Counter

### Pour l'évaluation finale des clusters, important de calculer un silhouette score sur embeddings initiaux (pour éviter déformations espace UMAP)
### + utilisation métrique cosine (la mieux adaptée aux embeddings de sentenceTransformers)
c_labels = labels_lora_kmeans
c_embs = embeddings_paraphrase_lora

custom_stop = ['livraison', 'commande', 'très', 'rapide']

# retirer le bruit HDBSCAN (-1)
df_viz = df_viz[df_viz.cluster != -1]

cluster_names = {}



for cluster_id in df_viz.cluster.unique():

    docs = df_viz[df_viz.cluster == cluster_id]["text"]
    #seuil= len(docs)/3  # filtre de fréquence d'occurence supplémentaire
    #custom_stop = Counter(docs)
    #custom_stop = [w for w, c in custom_stop.items() if c > seuil]
    #print(f"custom_stop: {custom_stop}")
    vectorizer = TfidfVectorizer(
        stop_words=stopwords.words("french") + custom_stop,
        #max_features=20,
        ngram_range=(1,2)
    )

    X = vectorizer.fit_transform(docs)

    scores = X.mean(axis=0).A1
    words = vectorizer.get_feature_names_out()

    ranking = sorted(
        zip(words, scores),
        key=lambda x: x[1],
        reverse=True
    )

    keywords = [w for w,_ in ranking[:5]]

    cluster_names[cluster_id] = " / ".join(keywords)

for cluster in cluster_names:
    print(f'Cluster: {cluster} \t size: {len(df_viz[df_viz['cluster'] == cluster])}')
    print(f'keywords: \t{cluster_names[cluster]}')

mask = c_labels != -1
if len(set(c_labels[mask])) > 1:
    score = silhouette_score(c_embs[mask], c_labels[mask], metric="cosine")   

#silhouette = silhouette_score(X_reduced_lora, labels_lora)
print(f'silhouette score: {score}')



Cluster: 5 	 size: 3684
keywords: 	colis / pièces / oscaro / bien / plus
Cluster: 2 	 size: 5265
keywords: 	conforme / parfait / efficace / rapidité / bien
Cluster: 0 	 size: 3500
keywords: 	colis / oscaro / pièces / plus / pièce
Cluster: 7 	 size: 4607
keywords: 	prix / pièces / conforme / bien / produit
Cluster: 4 	 size: 2948
keywords: 	conforme / bien / produit / parfait / rapidité
Cluster: 6 	 size: 2974
keywords: 	conforme / produit / prix / bien / produit conforme
Cluster: 1 	 size: 4343
keywords: 	colis / pièces / bien / conforme / produit
Cluster: 8 	 size: 3438
keywords: 	conforme / bien / prix / pièces / produit
Cluster: 3 	 size: 4450
keywords: 	conforme / produit / prix / bien / pièces
Cluster: 9 	 size: 3052
keywords: 	conforme / prix / produit / pièces / bien
silhouette score: -0.09509706497192383


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def get_representative_texts(embeddings, corpus, labels, top_k=3):
    cluster_repr = {}
    
    for cluster_id in set(labels):
        if cluster_id == -1:
            continue
        
        idx = [i for i in range(len(labels)) if labels[i] == cluster_id]
        cluster_emb = embeddings[idx]
        
        centroid = cluster_emb.mean(axis=0).reshape(1, -1)
        sim = cosine_similarity(cluster_emb, centroid).ravel()
        
        top_idx = np.argsort(sim)[-top_k:][::-1]
        
        cluster_repr[cluster_id] = [corpus[idx[i]] for i in top_idx]
    
    return cluster_repr

In [23]:
cluster_repr = get_representative_texts(embeddings_paraphrase_lora, df['commentaire'].to_list(), labels_lora_kmeans)
for i in cluster_repr:
    print(f'cluster {i}: {cluster_repr[i]}')

cluster 0: ["Bizarre de ne vendre qu'un seul vérin de coffre, car quand un fonctionne cela suffit à maintenir le coffre ouvert. Donc on ne peut pas savoir si il y en a un qui ne fonctionne pas.On ne le sait que lorsque les deux sont HS.", 'J ai commende 2 pièces livrable à une adresse de livraison mais les 2 livrés ( en 2 livraisons a une autre adresse . A quoi sert le choix de l adresse de livraison svp Toutes les conditions de ma commende sont sur votre site', 'Le produit correspond à ma demande. Cependant le colis est arrivé avec le carton en mauvais état. On a contrôlé à la réception. Je mets une réserve lors de le mise en route du radiateur. J espère qu il n y aura pas de fuite.']
cluster 1: ["Tout s'est très bien déroulé du moment de la commande jusqu'à la livraison et le retrait dans le relais colis ! Parfait !", "J'ai payé pour une livraison express qui a été livrée 1 semaine après la date de commande.m au lieux des deux jours annoncés.", 'On donne nos plaques pour avoir les bo

## Enregistrement des embeddings et des labels, des UMAP et KMeans

In [22]:
reducer_3d = UMAP(n_components=3, n_neighbors=15, min_dist=0, metric="cosine")
embeddings_paraphrase_3d = reducer_3d.fit_transform(embeddings_paraphrase)
embeddings_paraphrase_lora_3d = reducer_3d.fit_transform(embeddings_paraphrase_lora)

In [23]:

np.savez(f"{temp_save}/labels_embeddings_paraphrase_hdbscan_3d.npz", embeddings=embeddings_paraphrase_3d, labels=labels_nolora_agglo_postprocess)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_kmeans_3d.npz", embeddings=embeddings_paraphrase_3d, labels=labels_nolora_kmeans)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_lora_hdbscan_3d.npz", embeddings=embeddings_paraphrase_lora_3d, labels=labels_lora_agglo_postprocess)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_lora_kmeans_3d.npz", embeddings=embeddings_paraphrase_lora_3d, labels=labels_lora_kmeans)


In [24]:
np.savez(f"{temp_save}/labels_embeddings_paraphrase_hdbscan.npz", embeddings=embeddings_paraphrase, labels=labels_nolora_agglo_postprocess)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_kmeans.npz", embeddings=embeddings_paraphrase, labels=labels_nolora_kmeans)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_lora_hdbscan.npz", embeddings=embeddings_paraphrase_lora, labels=labels_lora_agglo_postprocess)
np.savez(f"{temp_save}/labels_embeddings_paraphrase_lora_kmeans.npz", embeddings=embeddings_paraphrase_lora, labels=labels_lora_kmeans)



In [25]:
from sklearn.cluster import HDBSCAN
import umap
import joblib

# export UMAP pipeline paraphrase + umap + hdbscan + agglo + postprocess
reducer_paraphrase_hdbscan = umap.UMAP(
    n_neighbors=best_umap_params_nolora[0],
    min_dist=best_umap_params_nolora[1],
    n_components=best_umap_params_nolora[2],
    metric="cosine",
    random_state=42
)
reducer_paraphrase_hdbscan.fit_transform(embeddings_paraphrase)
reducer_paraphrase_hdbscan.embedding_ = reducer_paraphrase_hdbscan.embedding_.astype("float32")
reducer_paraphrase_hdbscan._raw_data = reducer_paraphrase_hdbscan._raw_data.astype("float32")
reducer_paraphrase_hdbscan.graph_ = None
joblib.dump(reducer_paraphrase_hdbscan, f"{temp_save}/umap_paraphrase_hdbscan.pkl", compress=3)
#umap.utils.save_umap(reducer_paraphrase_hdbscan, "umap_model.umap")

# export UMAP pipeline paraphrase + umap + kmeans
reducer_paraphrase_kmeans = umap.UMAP(
    n_neighbors=best_umap_params_kmeans_nolora[0],
    min_dist=best_umap_params_kmeans_nolora[1],
    n_components=best_umap_params_kmeans_nolora[2],
    metric="cosine",
    random_state=42
)
reducer_paraphrase_kmeans.fit_transform(embeddings_paraphrase)
reducer_paraphrase_kmeans.embedding_ = reducer_paraphrase_kmeans.embedding_.astype("float32")
reducer_paraphrase_kmeans._raw_data = reducer_paraphrase_kmeans._raw_data.astype("float32")
reducer_paraphrase_kmeans.graph_ = None
joblib.dump(reducer_paraphrase_kmeans, f"{temp_save}/umap_paraphrase_kmeans.pkl", compress=3)

# export UMAP pipeline paraphrase + lora + umap + hdbscan + agglo + postprocess
reducer_paraphrase_lora_hdbscan = umap.UMAP(
    n_neighbors=best_umap_params_lora[0],
    min_dist=best_umap_params_lora[1],
    n_components=best_umap_params_lora[2],
    metric="cosine",
    random_state=42
)
reducer_paraphrase_lora_hdbscan.fit_transform(embeddings_paraphrase_lora)
reducer_paraphrase_lora_hdbscan.embedding_ = reducer_paraphrase_lora_hdbscan.embedding_.astype("float32")
reducer_paraphrase_lora_hdbscan._raw_data = reducer_paraphrase_lora_hdbscan._raw_data.astype("float32")
reducer_paraphrase_lora_hdbscan.graph_ = None
joblib.dump(reducer_paraphrase_lora_hdbscan, f"{temp_save}/umap_paraphrase_lora_hdbscan.pkl", compress=3)

# export UMAP pipeline paraphrase + lora + umap + kmeans
reducer_paraphrase_lora_kmeans = umap.UMAP(
    n_neighbors=best_umap_params_kmeans_lora[0],
    min_dist=best_umap_params_kmeans_lora[1],
    n_components=best_umap_params_kmeans_lora[2],
    metric="cosine",
    random_state=42
)
reducer_paraphrase_lora_kmeans.fit_transform(embeddings_paraphrase_lora)
reducer_paraphrase_lora_kmeans.embedding_ = reducer_paraphrase_lora_kmeans.embedding_.astype("float32")
reducer_paraphrase_lora_kmeans._raw_data = reducer_paraphrase_lora_kmeans._raw_data.astype("float32")
reducer_paraphrase_lora_kmeans.graph_ = None
joblib.dump(reducer_paraphrase_lora_kmeans, f"{temp_save}/umap_paraphrase_lora_kmeans.pkl", compress=3)

/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/robin/projets/Supply-Chain-Reviews/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


['data_2epoch/umap_paraphrase_lora_kmeans.pkl']

In [26]:
from sklearn.cluster import KMeans
clusterer_lora = KMeans(n_clusters=10, random_state=42, n_init="auto")
clusterer_lora.fit_predict(X_reduced_lora_kmeans)
joblib.dump(clusterer_lora, f"{temp_save}/kmeans_paraphrase_lora.pkl")

clusterer_nolora = KMeans(n_clusters=10, random_state=42, n_init="auto")
clusterer_nolora.fit_predict(X_reduced_nolora_kmeans)
joblib.dump(clusterer_nolora, f"{temp_save}/kmeans_paraphrase.pkl")


['data_2epoch/kmeans_paraphrase.pkl']